# LeetCode #1139 — Largest 1-Bordered Square: Hình vuông có viền toàn số 1

**Medium · Python 3 · Matrix, Prefix Runs, Dynamic Programming, Enumeration**  
[Đề LeetCode](https://leetcode.com/problems/largest-1-bordered-square/) · [Đối chiếu constraints](https://leetcode.ca/2019-01-12-1139-Largest-1-Bordered-Square/)

```python
class Solution:
    def largest1BorderedSquare(self, grid: List[List[int]]) -> int:
        ...
```

Ta sẽ kiểm tra các hình vuông từ cách đơn giản, sau đó học **DP đếm số 1 liên tục** theo hàng và cột để kiểm tra một viền trong O(1). Đề hỏi **diện tích (`side²`)**, không phải độ dài cạnh.

## 1. Mục tiêu cần đạt

1. Biết thao tác mảng 2 chiều: `grid[r][c]`, số hàng `R`, số cột `C`.
2. Phân biệt **các ô trên viền** với **các ô bên trong** hình vuông.
3. Viết hàm brute force để kiểm tra bốn cạnh bằng vòng lặp.
4. Suy ra `left[r][c]` và `up[r][c]` đếm số 1 liên tục.
5. Dùng 4 điều kiện để kiểm tra viền kích thước `k` trong O(1).
6. Phân tích O(R*C*min(R,C)), chạy test đối chiếu và tự làm bài tập.

## 2. Đề bài và ví dụ

Cho ma trận `grid` chỉ gồm `0` và `1`. Hãy tìm **diện tích (số ô) của hình vuông lớn nhất** có **toàn bộ các ô trên biên bằng `1`**. Các ô nằm **bên trong** được phép là `0`. Nếu không có ô `1` nào để tạo hình vuông thì trả `0`.

**Ví dụ 1:**

```text
1 1 1
1 0 1      →  hình vuông 3×3 có viền toàn 1, diện tích 9
1 1 1
```

**Ví dụ 2:** `[[1,1,0,0]]` ⇒ chỉ có hình vuông `1×1`, trả `1`.

**Ràng buộc đề:** `1 <= R,C <= 100`; `grid[i][j]` là `0` hoặc `1`.

In [1]:
GRID_EXAMPLE = [[1,1,1],[1,0,1],[1,1,1]]
for row in GRID_EXAMPLE:
    print(' '.join(map(str,row)))
print('Ô giữa:', GRID_EXAMPLE[1][1], '(không bắt buộc bằng 1)')
assert GRID_EXAMPLE[1][1]==0

1 1 1
1 0 1
1 1 1
Ô giữa: 0 (không bắt buộc bằng 1)


## 3. Python cơ bản: mảng 2 chiều, `len`, `range`

- `R=len(grid)` là số hàng.
- `C=len(grid[0])` là số cột (đề đảm bảo ma trận không rỗng).
- `grid[r][c]` lấy phần tử tại hàng `r`, cột `c` (đều 0-based).
- `for r in range(R)` duyệt các hàng, `for c in range(C)` duyệt các cột.

In [2]:
R,C=len(GRID_EXAMPLE),len(GRID_EXAMPLE[0])
print('R,C =',R,C)
for r in range(R):
    for c in range(C):
        print(f'grid[{r}][{c}]={GRID_EXAMPLE[r][c]}',end='; ')
    print()
assert (R,C)==(3,3)

R,C = 3 3
grid[0][0]=1; grid[0][1]=1; grid[0][2]=1; 
grid[1][0]=1; grid[1][1]=0; grid[1][2]=1; 
grid[2][0]=1; grid[2][1]=1; grid[2][2]=1; 


## 4. Một hình vuông được xác định bằng góc và cạnh

Nếu chọn **góc trên trái** `(top,left)` và độ dài cạnh `side=k`, ta có:

- góc dưới phải `(bottom,right)=(top+k-1,left+k-1)`;
- hàng trên `top`, hàng dưới `bottom`;
- cột trái `left`, cột phải `right`.

Chỉ hợp lệ nếu `0<=bottom<R` và `0<=right<C`. Diện tích cần trả là `k*k`.

In [3]:
top,left,k=0,0,3
bottom,right=top+k-1,left+k-1
print('Góc trên trái:',(top,left),'| dưới phải:',(bottom,right))
print('Cạnh:',k,'| diện tích:',k*k)
assert (bottom,right)==(2,2) and k*k==9

Góc trên trái: (0, 0) | dưới phải: (2, 2)
Cạnh: 3 | diện tích: 9


## 5. Kiểm tra một viền bằng bốn vòng `all`

`all(condition for x in iterable)` trả `True` nếu **mọi** phần tử thỏa điều kiện. Viền có 4 cạnh: hai hàng ngang và hai cột dọc. Ở bốn góc, ô được kiểm tra hai lần nhưng vẫn đúng.

Trước hết viết hàm nhỏ để **tự tin về định nghĩa**, sau đó mới giải toàn bài.

In [4]:
def has_one_border(grid,top,left,side):
    bottom=top+side-1
    right=left+side-1
    R,C=len(grid),len(grid[0])
    if top<0 or left<0 or bottom>=R or right>=C:
        return False
    return (all(grid[top][c]==1 for c in range(left,right+1))
            and all(grid[bottom][c]==1 for c in range(left,right+1))
            and all(grid[r][left]==1 for r in range(top,bottom+1))
            and all(grid[r][right]==1 for r in range(top,bottom+1)))

assert has_one_border(GRID_EXAMPLE,0,0,3)
assert not has_one_border(GRID_EXAMPLE,0,0,2)
assert not has_one_border(GRID_EXAMPLE,0,0,4)
print('Hàm kiểm tra viền: PASS')

Hàm kiểm tra viền: PASS


## 6. Vì sao ô chính giữa được phép là 0?

Với hình vuông `3×3` ở ví dụ 1, viền gồm 8 ô `1`; ô chính giữa `(1,1)` không nằm trên cạnh nào. Nếu đòi **toàn bộ** 9 ô là 1, bạn đang giải một bài khác (largest all-one square).

In [5]:
print('Mọi ô đều 1?',all(v==1 for row in GRID_EXAMPLE for v in row))
print('Viền 3x3 toàn 1?',has_one_border(GRID_EXAMPLE,0,0,3))
assert not all(v==1 for row in GRID_EXAMPLE for v in row)
assert has_one_border(GRID_EXAMPLE,0,0,3)

Mọi ô đều 1? False
Viền 3x3 toàn 1? True


## 7. Cách trực giác: thử mọi góc và mọi cạnh

Tại mỗi góc trên trái `(top,left)`, cạnh có thể là `1..min(R-top,C-left)`. Kiểm tra 4 cạnh; lưu `best` là **độ dài cạnh lớn nhất** tìm được, cuối cùng trả `best²`.

Cách này chậm hơn tối ưu nhưng dễ chứng minh đúng, dùng làm **oracle** (đáp án tham khảo độc lập) cho test ngẫu nhiên.

In [6]:
def largest_square_bruteforce(grid):
    R,C=len(grid),len(grid[0])
    best=0
    for top in range(R):
        for left in range(C):
            max_side=min(R-top,C-left)
            for side in range(1,max_side+1):
                if has_one_border(grid,top,left,side):
                    best=max(best,side)
    return best*best

assert largest_square_bruteforce(GRID_EXAMPLE)==9
assert largest_square_bruteforce([[1,1,0,0]])==1
assert largest_square_bruteforce([[0]])==0
print('Brute Force: PASS')

Brute Force: PASS


## 8. Trace brute force từng kích thước

Với bảng 3×3 mẫu, kích thước 1 tại `(0,0)` hợp lệ, kích thước 2 tại `(0,0)` **không** hợp lệ vì có `0` ở `(1,1)` trên viền 2×2; kích thước 3 lại **hợp lệ** vì `(1,1)` trở thành nội thất. Vì thế **không** được dừng sớm đơn giản sau khi thấy một kích thước nhỏ thất bại.

In [7]:
for side in [1,2,3]:
    print(f'Góc (0,0), cạnh={side}, viền toàn 1? {has_one_border(GRID_EXAMPLE,0,0,side)}')
assert [has_one_border(GRID_EXAMPLE,0,0,k) for k in (1,2,3)]==[True,False,True]

Góc (0,0), cạnh=1, viền toàn 1? True
Góc (0,0), cạnh=2, viền toàn 1? False
Góc (0,0), cạnh=3, viền toàn 1? True


## 9. Nút thắt: nhiều hình vuông cùng kiểm tra lại một đoạn

Với mỗi góc ta có tới `L=min(R,C)` cạnh; kiểm tra một viền cạnh `k` tốn **O(k)**. Tổng thời gian tệ nhất là **O(R*C*L²)**, vì `1+2+...+L=O(L²)`. Khi `R=C=100`, ta nên tránh quét lại các hàng/cột cho từng hình vuông.

Mấu chốt: **tính trước số 1 liên tiếp** rồi kiểm tra mỗi cạnh chỉ bằng một phép so sánh.

## 10. Kiến thức nền: bài toán nhỏ «đếm dãy số 1 liên tiếp»

Với hàng `[1,1,0,1,1,1]`, nếu `left[c]` ghi số chữ số `1` liên tiếp kết thúc tại cột `c`, ta có:

```text
grid: 1 1 0 1 1 1
left: 1 2 0 1 2 3
```

Công thức:

- Nếu `grid[r][c]==0` → `left[r][c]=0`.
- Nếu `grid[r][c]==1` → `left[r][c]=1+left[r][c-1]` (nếu c>0, ngược lại `1`).

Đây là **dynamic programming (DP)**: dùng kết quả của ô liền trước thay vì đếm lại từ đầu.

In [8]:
row=[1,1,0,1,1,1]
runs=[]
for x in row:
    runs.append((runs[-1] if runs else 0)+1 if x==1 else 0)
print('hàng:',row)
print('số 1 liên tục kết thúc tại vị trí:',runs)
assert runs==[1,2,0,1,2,3]

hàng: [1, 1, 0, 1, 1, 1]
số 1 liên tục kết thúc tại vị trí: [1, 2, 0, 1, 2, 3]


## 11. Làm tương tự theo chiều dọc: `up[r][c]`

`up[r][c]` là số ô `1` liên tiếp **đi lên** tính từ `(r,c)` (gồm chính ô đó). Nếu ô là `0`, `up=0`; nếu là `1`, `up=1 + up[r-1][c]`.

Ta dùng 2 bảng **`left` và `up`** với kích thước R×C, duyệt từ trên xuống, trái sang phải. Chỉ một lượt `O(R*C)`.

In [9]:
def build_runs(grid):
    R,C=len(grid),len(grid[0])
    left=[[0]*C for _ in range(R)]
    up=[[0]*C for _ in range(R)]
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 1:
                left[r][c] = 1 + (left[r][c-1] if c>0 else 0)
                up[r][c] = 1 + (up[r-1][c] if r>0 else 0)
    return left,up

LEFT,UP=build_runs(GRID_EXAMPLE)
print('left:')
for row in LEFT:print(row)
print('up:')
for row in UP:print(row)
assert LEFT==[[1,2,3],[1,0,1],[1,2,3]]
assert UP==[[1,1,1],[2,0,2],[3,1,3]]

left:
[1, 2, 3]
[1, 0, 1]
[1, 2, 3]
up:
[1, 1, 1]
[2, 0, 2]
[3, 1, 3]


## 12. Bước đột phá: kiểm tra viền bằng *4 phép so sánh*

Với hình vuông cạnh `k`, góc trên trái `(top,left_col)` và dưới phải `(bottom,right)`:

| Cạnh cần toàn `1` | Điều kiện |
|---|---|
| Cạnh trên | `left[top][right] >= k` |
| Cạnh dưới | `left[bottom][right] >= k` |
| Cạnh trái | `up[bottom][left_col] >= k` |
| Cạnh phải | `up[bottom][right] >= k` |

Nếu cả bốn đúng, toàn bộ viền là 1. Không cần duyệt từng ô của viền nữa.

**Ví dụ:** với `(top,left_col)=(0,0)`, `bottom=right=2`, `k=3`: các bảng đều trả giá trị 3 tại bốn điểm cần tra.

In [10]:
def has_one_border_dp(left,up,top,left_col,side):
    bottom=top+side-1
    right=left_col+side-1
    return (left[top][right]>=side and left[bottom][right]>=side
            and up[bottom][left_col]>=side and up[bottom][right]>=side)

print('4 số đo viền 3×3:',LEFT[0][2],LEFT[2][2],UP[2][0],UP[2][2])
assert has_one_border_dp(LEFT,UP,0,0,3)
assert not has_one_border_dp(LEFT,UP,0,0,2)

4 số đo viền 3×3: 3 3 3 3


## 13. Mô phỏng bằng tay: tại sao 2×2 sai nhưng 3×3 đúng?

Bảng ví dụ:

```text
grid       left       up
1 1 1      1 2 3      1 1 1
1 0 1      1 0 1      2 0 2
1 1 1      1 2 3      3 1 3
```

- `k=2`, góc `(0,0)`: `left[1][1]=0`, `up[1][1]=0` ⇒ có cạnh chứa `0` ⇒ loại.
- `k=3`, góc `(0,0)`: cả 4 giá trị ≥3 ⇒ nhận; diện tích = 9.

Đó là lý do cần **thử các kích thước độc lập**, không giả định một hình vuông lớn hợp lệ thì mọi hình vuông con cùng góc cũng hợp lệ.

In [11]:
for k in [1,2,3]:
    top=0;left_col=0
    right=left_col+k-1;bottom=top+k-1
    checks=(LEFT[top][right],LEFT[bottom][right],UP[bottom][left_col],UP[bottom][right])
    print(f'k={k}, số đo 4 cạnh={checks}, hợp lệ={all(x>=k for x in checks)}')

k=1, số đo 4 cạnh=(1, 1, 1, 1), hợp lệ=True
k=2, số đo 4 cạnh=(2, 0, 2, 0), hợp lệ=False
k=3, số đo 4 cạnh=(3, 3, 3, 3), hợp lệ=True


## 14. Lời giải tối ưu: thử cạnh **từ lớn xuống nhỏ**

Sau khi tính `left`, `up`, ta xét `side=min(R,C), ..., 1`. Với mỗi `side`, thử mọi vị trí `(top,left_col)` nằm trọn trong grid.

**Ngay khi gặp** một hình vuông hợp lệ ở cạnh `side`, trả `side*side`. Vì đã xét tất cả cạnh lớn hơn và không có hình nào hợp lệ, đây chính là diện tích lớn nhất.

In [12]:
def largest_square_dp(grid):
    R,C=len(grid),len(grid[0])
    left,up=build_runs(grid)
    for side in range(min(R,C),0,-1):
        for top in range(R-side+1):
            for left_col in range(C-side+1):
                if has_one_border_dp(left,up,top,left_col,side):
                    return side*side
    return 0

assert largest_square_dp(GRID_EXAMPLE)==9
assert largest_square_dp([[1,1,0,0]])==1
assert largest_square_dp([[0]])==0
print('DP + kiểm tra 4 cạnh: PASS')

DP + kiểm tra 4 cạnh: PASS


## 15. Code đúng format LeetCode

Code sau **không phụ thuộc các hàm demo**; có thể dán trực tiếp vào bài #1139 trên LeetCode. Nó dùng `left` và `up` theo chỉ số của ma trận đầu vào.

In [13]:
from typing import List

class Solution:
    def largest1BorderedSquare(self, grid: List[List[int]]) -> int:
        R,C=len(grid),len(grid[0])
        left=[[0]*C for _ in range(R)]
        up=[[0]*C for _ in range(R)]
        for r in range(R):
            for c in range(C):
                if grid[r][c] == 1:
                    left[r][c]=1+(left[r][c-1] if c>0 else 0)
                    up[r][c]=1+(up[r-1][c] if r>0 else 0)
        for k in range(min(R,C),0,-1):
            for top in range(R-k+1):
                for left_col in range(C-k+1):
                    bottom=top+k-1
                    right=left_col+k-1
                    if (left[top][right]>=k and left[bottom][right]>=k
                            and up[bottom][left_col]>=k and up[bottom][right]>=k):
                        return k*k
        return 0

assert Solution().largest1BorderedSquare(GRID_EXAMPLE)==9
assert Solution().largest1BorderedSquare([[1,1,0,0]])==1
print('LeetCode Solution: PASS')

LeetCode Solution: PASS


## 16. TRACE thuật toán: vì sao chọn ra vị trí ấy?

Trace in bảng `left/up`, duyệt `k` từ lớn xuống, ghi mỗi hình vuông thử ở đầu. Bản nộp không cần in debug; chỉ dùng trace để theo dõi **điều kiện kiểm tra từng cạnh**, đặc biệt khi điểm bắt đầu/điểm kết thúc bị lệch chỉ số.

In [14]:
def trace_largest_square(grid):
    R,C=len(grid),len(grid[0])
    left,up=build_runs(grid)
    print('grid:')
    for row in grid:print(row)
    print('left:')
    for row in left:print(row)
    print('up:')
    for row in up:print(row)
    for k in range(min(R,C),0,-1):
        print('Thử side=',k)
        for top in range(R-k+1):
            for l in range(C-k+1):
                bottom,right=top+k-1,l+k-1
                checks=(left[top][right],left[bottom][right],up[bottom][l],up[bottom][right])
                valid=all(x>=k for x in checks)
                print(f'  top={top}, left={l}, số đo={checks}, hợp lệ={valid}')
                if valid:return k*k
    return 0

assert trace_largest_square(GRID_EXAMPLE)==9

grid:
[1, 1, 1]
[1, 0, 1]
[1, 1, 1]
left:
[1, 2, 3]
[1, 0, 1]
[1, 2, 3]
up:
[1, 1, 1]
[2, 0, 2]
[3, 1, 3]
Thử side= 3
  top=0, left=0, số đo=(3, 3, 3, 3), hợp lệ=True


## 17. Chứng minh thuật toán đúng

**Bất biến khi xây bảng:** `left[r][c]` chính xác là số ô 1 liên tiếp kết thúc tại `(r,c)` theo hàng, vì ô 0 → 0; ô 1 → 1 cộng chuỗi liền bên trái đã được tính. Lý luận tương tự cho `up` theo cột.

**Tương đương:** một viền cạnh `k` hợp lệ khi và chỉ khi bốn đoạn cạnh dài `k` toàn 1; bốn số đo `left/up` kiểm tra đúng từng đoạn ấy, không thiếu cũng không thừa.

**Tối ưu:** duyệt `k` giảm dần; cạnh lớn hơn đã được kiểm hết vị trí trước khi xét cạnh nhỏ hơn. Khi gặp đầu tiên hợp lệ, diện tích `k²` chắc chắn lớn nhất. Nếu không thấy bất kỳ cạnh 1 nào thì trả 0 đúng đề.

## 18. Phân tích Time / Space Complexity

Gọi `R` = số hàng, `C` = số cột, `L=min(R,C)`.

- Xây hai bảng `left,up`: **O(R*C)** thời gian và **O(R*C)** bộ nhớ.
- Có tối đa `L` độ dài cạnh; với mỗi cạnh thử tối đa `R*C` vị trí; mỗi phép kiểm tra chỉ 4 so sánh: **O(R*C*L)** thời gian.
- Tổng thời gian: **O(R*C*min(R,C))**; bộ nhớ phụ **O(R*C)**.
- Brute Force 4 cạnh mất O(k) mỗi hình, cộng lại **O(R*C*L²)** thời gian trong worst-case; bộ nhớ phụ O(1) (không kể input).

Một hình 100×100 cần tối đa 1.000.000 lần kiểm tra O(1) theo cận thô, hợp lý cho Python.

## 19. Test ví dụ, biên, lưới không vuông và bên trong chứa 0

Lưu ý: `1×1` cũng là một **hình vuông có viền**. Một hàng dài hoặc một cột dài không thể tạo cạnh lớn hơn 1.

In [15]:
samples = [
    ('vòng 3x3',[[1,1,1],[1,0,1],[1,1,1]],9),
    ('hàng đơn',[[1,1,0,0]],1),
    ('không có 1',[[0]],0),
    ('một ô 1',[[1]],1),
    ('vuông 4x4 toàn 1',[[1]*4 for _ in range(4)],16),
    ('vuông 4x4 rỗng giữa',[[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]],16),
    ('hàng có 0 giữa',[[1,0,1,1,1]],1),
    ('cột',[[1],[1],[0],[1]],1),
    ('toàn 0',[[0,0],[0,0]],0),
    ('không vuông',[[1,1,1],[1,1,1]],4),
]
for name,grid,expected in samples:
    actual=Solution().largest1BorderedSquare(grid)
    assert actual==expected,(name,actual,expected)
    print(f'{name}: PASS ({actual})')

vòng 3x3: PASS (9)
hàng đơn: PASS (1)
không có 1: PASS (0)
một ô 1: PASS (1)
vuông 4x4 toàn 1: PASS (16)
vuông 4x4 rỗng giữa: PASS (16)
hàng có 0 giữa: PASS (1)
cột: PASS (1)
toàn 0: PASS (0)
không vuông: PASS (4)


## 20. Kiểm thử đối chiếu ngẫu nhiên với brute force

Viết hai cách giải độc lập là một cách kiểm chứng mạnh hơn tự test mỗi cách với vài ví dụ. Sinh ma trận nhỏ ngẫu nhiên, so `Solution` với brute force. Chọn kích thước 1..5 để việc quét tất cả hình vuông không tốn nhiều thời gian.

In [16]:
import random
random.seed(1139)
for trial in range(350):
    R=random.randint(1,5)
    C=random.randint(1,5)
    grid=[[random.randrange(2) for _ in range(C)] for _ in range(R)]
    reference=largest_square_bruteforce(grid)
    fast=Solution().largest1BorderedSquare(grid)
    assert fast==reference,(trial,grid,reference,fast)
print('350 lưới ngẫu nhiên 1..5: PASS')

350 lưới ngẫu nhiên 1..5: PASS


## 21. Kiểm tra kích thước cực đại theo constraints

Test 100×100 toàn 1: đáp án `100² = 10000`; test 100×100 toàn 0: đáp án `0`. Đây không phải benchmark thời gian chuẩn, mà là kiểm tra lỗi chỉ số hoặc độ phức tạp bất thường.

In [17]:
ones=[[1]*100 for _ in range(100)]
zeros=[[0]*100 for _ in range(100)]
assert Solution().largest1BorderedSquare(ones)==10000
assert Solution().largest1BorderedSquare(zeros)==0
print('100×100 toàn 1/toàn 0: PASS')

100×100 toàn 1/toàn 0: PASS


## 22. Các lỗi phổ biến (code sai → lý do → cách sửa)

1. **Trả `side` thay vì `side*side`:** đề yêu cầu *diện tích*.
2. **Kiểm tra cả nội thất:** loại sai hình vuông có biên toàn 1 nhưng tâm là 0.
3. **Nhầm `left[top][right]` với `left[top][left_col]`:** để kiểm tra cả hàng trên dài k, phải đếm chuỗi kết thúc tại mép **phải**.
4. **Không reset DP khi gặp 0:** dãy số 1 bị đứt, giá trị đếm phải trở về 0.
5. **`[[0]*C]*R` tạo hàng dùng chung:** dùng comprehension `[[0]*C for _ in range(R)]`.
6. **Dừng vì cạnh 2 sai ở một góc rồi kết luận cạnh 3 sai:** ví dụ `grid` có tâm 0 đã chứng minh điều ngược lại.

In [18]:
wrong=[[0]*3]*3
wrong[0][0]=7
print('Tạo bảng sai (các hàng cùng đổi):',wrong)
correct=[[0]*3 for _ in range(3)]
correct[0][0]=7
print('Tạo bảng đúng:',correct)
assert wrong[1][0]==7 and correct[1][0]==0

Tạo bảng sai (các hàng cùng đổi): [[7, 0, 0], [7, 0, 0], [7, 0, 0]]
Tạo bảng đúng: [[7, 0, 0], [0, 0, 0], [0, 0, 0]]


## 23. Mẹo nhận dạng DP «đếm liên tiếp» và biến thể

Nếu bài cần kiểm tra **một đoạn toàn 1** rất nhiều lần, hãy thử tiền xử lý độ dài dãy 1 liên tiếp. Từ đó kiểm tra đoạn dài `k` bằng đúng một phép so sánh.

Biến thể có thể gặp:
- Hình chữ nhật có viền toàn 1: thử cả chiều cao và chiều rộng, vẫn dùng các bảng liên tiếp.
- Hình vuông **toàn bộ ô là 1**: bài khác, dùng DP `dp[r][c] = 1+min(top,left,diag)` cho kích thước hình vuông lớn nhất kết thúc ở `(r,c)`.
- Nếu cập nhật lưới nhiều lần, 2 bảng `left/up` tĩnh không còn đủ; cần cấu trúc dữ liệu động khác.

## 24. Bài tập tự luyện (từ dễ đến khó)

**Bài 1 — Tính tay:** với `[[1,1,0],[1,1,1]]`, diện tích hình vuông viền toàn 1 lớn nhất là bao nhiêu?

**Bài 2 — Dự đoán output:** `has_one_border([[1,1,1],[1,0,1],[1,1,1]],0,0,2)` và cùng hàm khi `side=3` trả gì?

**Bài 3 — TODO:** viết `exercise_runs(grid)` trả `(left,up)`; không gọi `build_runs`.

**Bài 4 — TODO:** viết `exercise_has_border(left,up,top,left_col,k)` bằng bốn điều kiện O(1).

**Bài 5 — TODO/mini challenge:** viết `exercise_largest_square(grid)` bằng DP và tìm cạnh từ lớn xuống. Nếu rảnh, sửa để trả thêm `(top,left_col,side)` của một hình tối ưu.

Các hàm TODO chỉ được **định nghĩa**, không chạy test tự động đến khi bạn hoàn thành.

In [19]:
def exercise_runs(grid):
    # TODO: tạo hai bảng R x C và cập nhật left/up từ trên xuống, trái sang phải
    pass

def exercise_has_border(left,up,top,left_col,k):
    # TODO: xác định bottom, right; kiểm tra 4 cạnh bằng >=k
    pass

def exercise_largest_square(grid):
    # TODO: dựng bảng; thử k từ L xuống 1; trả k*k hoặc 0
    pass

# Tự bật test sau khi đã làm:
# assert exercise_runs(GRID_EXAMPLE) == (LEFT,UP)
# assert exercise_has_border(LEFT,UP,0,0,3) is True
# assert exercise_largest_square(GRID_EXAMPLE) == 9

## 25. Gợi ý 3 mức (mở khi bí)

| Bài | Gợi ý ý tưởng | Gợi ý cú pháp | Gần-code |
|---|---|---|---|
| 1 | Quan sát góc 2×2 phía trái | Viền 4 ô bằng 1 | Cạnh lớn nhất 2 |
| 2 | Cạnh 2 đi qua ô tâm 0 | `all(...)` | Kết quả `False`, rồi `True` |
| 3 | Nếu ô 0 → hai bảng đều 0 | `[[0]*C for _ in range(R)]` | `left[r][c]=1+(left[r][c-1] if c else 0)` |
| 4 | Tra mỗi cạnh tại một điểm cuối | `bottom=top+k-1` | So 4 chỉ số trong bảng mục 12 |
| 5 | Kết quả là diện tích | `range(min(R,C),0,-1)` | Có hình hợp lệ ⇒ `return k*k` |

## 26. Đáp án tham khảo (chỉ đọc sau khi làm)

**Bài 1:** hình vuông `2×2` ở góc trên trái có 4 ô 1, nên diện tích **4**.

**Bài 2:** với cạnh 2 ⇒ **False**; cạnh 3 ⇒ **True**.

Các bài còn lại có code đáp án sau. Bài 5 mở rộng trả thêm vị trí nếu cần: `return (area, (top,left_col,k))`, nhưng không dùng kiểu trả này cho hàm submit gốc.

In [20]:
def answer_runs(grid):
    R,C=len(grid),len(grid[0])
    left=[[0]*C for _ in range(R)]
    up=[[0]*C for _ in range(R)]
    for r in range(R):
        for c in range(C):
            if grid[r][c]:
                left[r][c]=1+(left[r][c-1] if c else 0)
                up[r][c]=1+(up[r-1][c] if r else 0)
    return left,up

def answer_has_border(left,up,top,l,k):
    bottom,right=top+k-1,l+k-1
    return (left[top][right]>=k and left[bottom][right]>=k
            and up[bottom][l]>=k and up[bottom][right]>=k)

def answer_largest_square(grid):
    R,C=len(grid),len(grid[0])
    left,up=answer_runs(grid)
    for k in range(min(R,C),0,-1):
        for r in range(R-k+1):
            for c in range(C-k+1):
                if answer_has_border(left,up,r,c,k):
                    return k*k
    return 0

assert answer_runs(GRID_EXAMPLE)==(LEFT,UP)
assert answer_has_border(LEFT,UP,0,0,3)
assert answer_largest_square(GRID_EXAMPLE)==9
print('Bộ đáp án bài tập: PASS')

Bộ đáp án bài tập: PASS


## 27. Template nhớ nhanh & checklist LeetCode

```text
Bước 1: tính left[r][c] (số 1 liền bên trái + ô hiện tại)
Bước 2: tính up[r][c] (số 1 liền bên trên + ô hiện tại)
Bước 3: thử k từ min(R,C) giảm dần tới 1
Bước 4: thử (top,left_col) sao cho hình k×k nằm trong grid
Bước 5: 4 cạnh dài k đều là 1? → return k*k
Nếu không có: return 0
```

- [ ] Không yêu cầu các ô **nội thất** bằng 1.
- [ ] Kiểm tra đúng 4 cạnh bằng `left/up`.
- [ ] Dùng bảng DP có các hàng **độc lập**.
- [ ] Trả diện tích `k*k`, không phải `k`.
- [ ] Test một hàng, một cột, ma trận toàn 0 và lưới 100×100.